In [1]:
using Pkg, REopt, HiGHS, JSON, JuMP, CSV, DataFrames, Sockets, Formatting, Dates

# Replace with your API Key
ENV["NREL_DEVELOPER_API_KEY"] = "EMXhDJ1wZrilRcOmFMHNIOASzWog8RLtJIPr22ep"
Pkg.status()

[ Info: Precompiling REopt [d36ad4e8-d74a-4f7a-ace1-eaea049febf6] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Info: Skipping precompilation due to precompilable error. Importing REopt [d36ad4e8-d74a-4f7a-ace1-eaea049febf6].
└   exception = Error when precompiling module, potentially caused by a __precompile__(false) declaration in the module.

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
[ Info: Precompiling ArchGDAL [c9ce4bd3-c3d5-55b8-8973-c0e20141b8c3] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile_

Status `C:\Users\kprabaka\Documents\2026\rereopt\REopt-MPC\Project.toml`
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.1
  [59287772] Formatting v0.4.3
  [87dc4568] HiGHS v1.22.2
⌅ [682c06a0] JSON v0.21.4
  [4076af6c] JuMP v1.30.0
  [d36ad4e8] REopt v0.52.0 `https://github.com/NatLabRockies/REopt.jl.git#h2-tldrd-mpc`
  [6462fe0b] Sockets v1.11.0
Info Packages marked with ⌅ have new versions available but compatibility constraints restrict them from upgrading. To see why use `status --outdated`


In [2]:
# Paths
inputs_path = "inputs"
outputs_path = "outputs"

# Filenames
load_file = "flat_load_500kw.csv"
pv_AMY_prod_factors_file = "pv_prod_factor_2h.csv" 
energy_cost_file = "tou_energy_costs_2h.csv"
demand_cost_file = "tou_demand_costs_2h.csv"

# Load relevant csvs 
full_load = CSV.read(inputs_path * "/" * load_file, DataFrame)
pv_AMY_prod_factor = CSV.read(inputs_path * "/" * pv_AMY_prod_factors_file, DataFrame)
energy_cost = CSV.read(inputs_path * "/" * energy_cost_file, DataFrame)
demand_cost = CSV.read(inputs_path * "/" * demand_cost_file, DataFrame)

load = full_load[!,"loads_kW"]
energy_rate = energy_cost[!,"usd_per_kwh"]
demand_cost = demand_cost[!,"usd_per_kw"]

month_transition_timesteps = [744, 1416, 2160, 2880, 3624, 4344, 5088, 5832, 6552, 7296, 8016, 8760]

# SET THESE! #####################################################
looping_method = "drts" # Options: perfect, imperfect_1, drts

save_results = true
filename = "RTDS_demo_v1" 

reopt_resource_type = "actual_year" # Options: actual_year, 8760 (strings)
control_algorithm_resource_type = "actual_year" # Options: actual_year
new_year_starting_ts = 25

allow_export = false
include_climate_in_objective = false
###################################################################

# System sizes
pv_kw = 250 #1227 
bess_kw = 500 #772 
bess_kwh = 625 #3090 

# Storage efficiencies
soc_min_fraction = 0.2
internal_efficiency_fraction = 0.975
inverter_efficiency_fraction = 0.96
rectifier_efficiency_fraction = 0.96

charge_efficiency = rectifier_efficiency_fraction * internal_efficiency_fraction^0.5
discharge_efficiency = inverter_efficiency_fraction * internal_efficiency_fraction^0.5

pv_prod_factor = pv_AMY_prod_factor[!,"pv_prod_factor_2024"]
realized_pv_prod_factor = pv_AMY_prod_factor[!,"pv_prod_factor_2024"]

# Demand cost inputs structure for initial post
tou_demand_rates = [0.0, 0.0, 0.0, 0.0]
tou_demand_ratchet_time_steps = [[], []]
tou_previous_peak_demands = [0, 0]; 

rate_scenario = "tou"
heuristics_dispatch = false;

In [3]:
# UDP configuration parameters
const REMOTE_IP = "10.81.15.164"   # Must match listener IP
const REMOTE_PORT = 7777        # Must match listener port
const LOCAL_PORT = 10001 

function write_be_float64(io::IO, x::Float64)
    u = reinterpret(UInt64, x)
    write(io, hton(u))
end

function write_be_float32(io, x::Float32)
    u = reinterpret(UInt32, x)
    write(io, hton(u))   # convert to network byte order (big endian)
end

function read_be_float32(chunk::Vector{UInt8})
    u = ntoh(reinterpret(UInt32, chunk)[1])   # big-endian -> host order
    return reinterpret(Float32, u)
end

function unpack_float32_be(data::Vector{UInt8})
    n = length(data)
    if n % 4 != 0
        error("UDP payload length $n is not a multiple of 4 bytes")
    end

    values = Vector{Float32}(undef, n ÷ 4)
    for j in 1:length(values)
        i = 4*(j-1) + 1
        u = ntoh(reinterpret(UInt32, data[i:i+3])[1])
        values[j] = reinterpret(Float32, u)
    end
    return values
end

unpack_float32_be (generic function with 1 method)

In [4]:
post = Dict([
    ("Settings", Dict([
        ("include_climate_in_objective", include_climate_in_objective),
        ])
    ),
    ("Site", Dict([
        ("include_exported_elec_emissions_in_total", true),
        ])
    ),
    ("ElectricLoad", Dict([
        ("loads_kw", [])
        ])),
    ("ElectricTariff",Dict([ 
       ("energy_rates", []),
       ("tou_demand_rates", tou_demand_rates),
       ("tou_demand_ratchet_time_steps", tou_demand_ratchet_time_steps),
       ("tou_previous_peak_demands", tou_previous_peak_demands) 
        ])), 
    ("ElectricUtility", Dict([
        ("emissions_factor_series_lb_CO2_per_kwh", [0.0]),
        ("emissions_factor_series_lb_NOx_per_kwh", [0.0]),
        ("emissions_factor_series_lb_SO2_per_kwh", [0.0]),
        ("emissions_factor_series_lb_PM25_per_kwh", [0.0])
        ])
    ),
#     ("Financial", Dict([
#         ("CO2_cost_per_tonne", 100.0)
#         ]
#         )),
    ("PV", Dict([
        ("size_kw", pv_kw),
        ("production_factor_series", [])
        ]
        )), 
    ("ElectricStorage",Dict([
        ("size_kw", bess_kw), 
        ("size_kwh", bess_kwh), 
        ("charge_efficiency", charge_efficiency),
        ("discharge_efficiency", discharge_efficiency),
        ("soc_init_fraction", 0.2),
        ("soc_min_fraction", soc_min_fraction),
        ("capacity_based_per_ts_self_discharge_fraction", 0.0),
        ("soc_based_per_ts_self_discharge_fraction", 0.0),
        ("fixed_dispatch_series", nothing)
        ]))
    ]);

In [5]:
function get_tou_demand_time_steps(cost_array, ts_array)
    tou_demand_ts = []
    for cost in cost_array
        push!(tou_demand_ts, findall(x->x==cost, ts_array))
    end
    return tou_demand_ts
end

get_tou_demand_time_steps (generic function with 1 method)

In [6]:
function dispatch_method_1(result, r1_ts, r2_ts, bess_kwh, previous_soc, charge_efficiency, discharge_efficiency, hours_per_time_step)

    # Process control execution
    load_ts = result["ElectricLoad"]["load_series_kw"][1]
    if r2_ts == 0
        bess_dispatch_ts = result["ElectricStorage"]["storage_to_load_series_kw"][1] - 
                           result["PV"]["electric_to_storage_series_kw"][1] -  
                           result["ElectricUtility"]["electric_to_storage_series_kw"][1]
    else
        bess_dispatch_ts = result["ElectricStorage"]["storage_to_load_series_kw"][1] - 
                           result["PV"]["electric_to_storage_series_kw"][1] - 
                           result["Wind"]["electric_to_storage_series_kw"][1] - 
                           result["ElectricUtility"]["electric_to_storage_series_kw"][1]
    end
    
    # RE resources are greater than the load (resource 1 is prioritized)
    if (r1_ts + r2_ts) >= load_ts
        util_to_load = 0
        bess_to_load = 0

        # Resource 1 generates enough to serve load by itself
        if r1_ts >= load_ts
            r1_to_load = load_ts
            r1_excess = r1_ts - r1_to_load
            r2_to_load = 0
            r2_excess = r2_ts

        # Both resources are need to serve load
        else 
            r1_to_load = r1_ts
            r1_excess = 0
            r2_to_load = load_ts - r1_to_load
            r2_excess = r2_ts - r2_to_load
        end

        # Control signal - Battery discharge
        if bess_dispatch_ts >= 0
            r1_to_bess = 0
            r2_to_bess = 0
            util_to_bess = 0
            bess_soc = previous_soc

         # Control signal - Battery charge
        else
            bess_soc = min(1.0, previous_soc + (abs(bess_dispatch_ts) * charge_efficiency * hours_per_time_step) / bess_kwh)

            # Resource 1 generates enough to charge battery after serving load
            if r1_excess > abs(bess_dispatch_ts)
                r1_to_bess = abs(bess_dispatch_ts)
                r1_excess = r1_excess - abs(bess_dispatch_ts)
                r2_to_bess = 0
                util_to_bess = 0 

            # Resource 1 and 2 together generate enough to charge battery after serving load
            elseif (r1_excess + r2_excess) > abs(bess_dispatch_ts)
                r1_to_bess = r1_excess
                r1_excess = 0
                r2_to_bess = abs(bess_dispatch_ts) - r1_to_bess
                r2_excess = r2_excess - r2_to_bess
                util_to_bess = 0

            # Excess RE is insufficient to charge battery    
            else 
                r1_to_bess = r1_excess
                r1_excess = 0
                r2_to_bess = r2_excess
                r2_excess = 0
                util_to_bess = abs(bess_dispatch_ts) - r1_to_bess - r2_to_bess
            end
        end

    # RE resouces are not enough to meet load
    else 
        r1_to_load = r1_ts
        r1_excess = 0
        r1_to_bess = 0
        r2_to_load = r2_ts
        r2_excess = 0            
        r2_to_bess = 0
        unmet_load = load_ts - r1_ts - r2_ts

        # Control signal - Battery discharge
        if bess_dispatch_ts >= 0
            util_to_bess = 0

            # Optimal battery discharge signal can meet the load
            if bess_dispatch_ts >= unmet_load
                bess_to_load = unmet_load
                util_to_load = 0
                bess_soc = max(soc_min_fraction, previous_soc - ((unmet_load/discharge_efficiency) * hours_per_time_step) / bess_kwh)

            # Optimal battery dispatch signal cannot meet the load
            else 
                bess_to_load = bess_dispatch_ts
                util_to_load = unmet_load - bess_dispatch_ts
                bess_soc = max(soc_min_fraction, previous_soc - ((bess_dispatch_ts/discharge_efficiency) * hours_per_time_step) / bess_kwh)
            end

        # Control signal - Battery charge  
        else 
            util_to_load = unmet_load
            util_to_bess = abs(bess_dispatch_ts)
            bess_to_load = 0
            bess_soc = min(1.0, previous_soc + (abs(bess_dispatch_ts) * charge_efficiency * hours_per_time_step) / bess_kwh)
        end
    end

    return r1_to_load, r1_to_bess, r1_excess, r2_to_load, r2_to_bess, r2_excess, bess_to_load, bess_soc, util_to_load, util_to_bess
end

dispatch_method_1 (generic function with 1 method)

In [7]:
"""
Decompose aggregate power measurements into physical flows.

Inputs (kW):
- pv >= 0
- battery > 0 discharge, < 0 charge
- load <= 0 consumption
- grid > 0 import, < 0 export

Outputs (all >= 0):
- grid_serving_load
- grid_charging_battery
- pv_serving_load
- pv_charging_battery
- battery_serving_load
- exported_pv
"""
function split_power_flows(pv, battery, load, grid)

    # Convert to positive magnitudes
    load_demand     = max(-load, 0.0)
    batt_discharge  = max(battery, 0.0)
    batt_charge     = max(-battery, 0.0)
    grid_import     = max(grid, 0.0)

    # ----- LOAD SUPPLY -----

    # 1) PV → load
    pv_serving_load = min(pv, load_demand)

    remaining_load = load_demand - pv_serving_load

    # 2) Battery → load
    battery_serving_load = min(batt_discharge, remaining_load)

    remaining_load -= battery_serving_load

    # 3) Grid → load
    grid_serving_load = min(grid_import, remaining_load)

    # ----- BATTERY CHARGING -----

    # PV remaining after serving load
    remaining_pv = max(pv - pv_serving_load, 0.0)

    # 4) PV → battery
    pv_charging_battery = min(remaining_pv, batt_charge)

    remaining_pv -= pv_charging_battery
    remaining_batt_charge = batt_charge - pv_charging_battery

    # 5) Grid → battery
    remaining_grid_import = max(grid_import - grid_serving_load, 0.0)
    grid_charging_battery = min(remaining_grid_import, remaining_batt_charge)

    # ----- EXPORT -----

    # 6) Remaining PV exported
    exported_pv = max(remaining_pv, 0.0)

    return (
        grid_serving_load      = grid_serving_load,
        grid_charging_battery  = grid_charging_battery,
        pv_serving_load        = pv_serving_load,
        pv_charging_battery    = pv_charging_battery,
        battery_serving_load   = battery_serving_load,
        exported_pv            = exported_pv,
    )
end

split_power_flows

In [ ]:
# MPC Loop

hours_per_time_step = 1/120
horizon = 30 # e.g., 24 hour lookahead
stopping_ts = 120 # 8760 for full year run
starting_ts = 1
length_of_data = 240

# Running sum variables
energy_cost_kwh = 0
soc_init_fraction = 0.2
tou_previous_peak_demands = zeros(length(tou_demand_rates))
monthly_demand_cost = []

tou_peaks_by_month = []

# Full results from each MPC loop
results = []

# Executed dispatch results 
dispatch_results = Dict([
    ("PV", Dict([
        ("electric_to_load_series_kw", []),
        ("electric_to_storage_series_kw", []),
        ("electric_to_grid_series_kw", []),
        ("electric_curtailed_series_kw", [])
        ])
    ),
    ("ElectricStorage", Dict([
        ("storage_to_load_series_kw", []),
        ("soc_series_fraction", [])
        ])
    ),        
    ("ElectricUtility", Dict([
        ("electric_to_load_series_kw", []),
        ("electric_to_storage_series_kw", [])          
        ])
    ),
    ("ElectricLoad", Dict([
        ("load_series_kw", []),
        ])
    )
])

drts_signals = Dict([
    ("PV", Dict([
        ("Pset_pv", []),
        ("Qset_pv", [])
        ])
    ),
    ("Battery", Dict([
        ("Pset_bess", []),
        ("Qset_bess", []),
        ("BESS_soc", [])
        ])
    ),        
    ("Load", Dict([
        ("Pset_load", []),
        ("Qset_load", [])           
        ])
    ),
    ("Grid", Dict([
        ("P_grid", []),
        ("Q_grid", [])
        ])
    )
])

cost_series = []

if looping_method == "drts"
    const PERIOD_NS   = 30_000_000_000 # 30 sec
    
    #sock = UDPSocket()
    #bind(sock, ip"10.81.15.151", LOCAL_PORT)
    run_count = 0
    
    # First send time = now
    next_send_time = time_ns()
end

for idx in range(starting_ts, stop=stopping_ts)
    
    end_ts = idx+horizon-1
    if end_ts > length_of_data
        end_ts = length_of_data
    end
    
    # Update forecast for each run

    post["PV"]["production_factor_series"] = pv_prod_factor[idx:end_ts]
    post["ElectricTariff"]["energy_rates"] = energy_rate[idx:end_ts]
    post["ElectricLoad"]["loads_kw"] = load[idx:end_ts]

    post["ElectricStorage"]["soc_init_fraction"] = soc_init_fraction
        
    if rate_scenario == "tou"
        tou_demand_ratchet_time_steps = get_tou_demand_time_steps(tou_demand_rates, demand_cost[idx:end_ts])
        
#         post["ElectricTariff"]["tou_demand_rates"] = tou_demand_rates # Set earlier, doesn't change by loop
        post["ElectricTariff"]["tou_demand_ratchet_time_steps"] = tou_demand_ratchet_time_steps
        post["ElectricTariff"]["tou_previous_peak_demands"] = tou_previous_peak_demands
    end
    
    if heuristics_dispatch
        post["ElectricStorage"]["fixed_dispatch_series"] = bess_dispatch[idx:end_ts]
    end

    # Run optimization
    model = Model(optimizer_with_attributes(HiGHS.Optimizer, "output_flag" => false, "log_to_console" => false))
    result = run_mpc(model, post)
    
    # Save full set of results per MPC loop 
    push!(results, result) 
    
    # Get results needed for the next iteration of the MPC algorithm
    if looping_method == "perfect"
        
        r1_to_load = result["PV"]["electric_to_load_series_kw"][1]
        r1_to_bess = result["PV"]["electric_to_storage_series_kw"][1]
        # Not sure if REopt MPC ever sends anything to grid
        # Currently assumes all excess is exported or curtailed (export benefits not calculated though)
        r1_excess = result["PV"]["electric_to_grid_series_kw"][1] + result["PV"]["electric_curtailed_series_kw"][1]
        
        bess_to_load = result["ElectricStorage"]["storage_to_load_series_kw"][1]
        bess_soc = result["ElectricStorage"]["soc_series_fraction"][1]
        util_to_load = result["ElectricUtility"]["electric_to_load_series_kw"][1]
        util_to_bess = result["ElectricUtility"]["electric_to_storage_series_kw"][1]

        grid_power = max(result["ElectricUtility"]["electric_to_load_series_kw"][1] + 
                         result["ElectricUtility"]["electric_to_storage_series_kw"][1], 0)      
        
    elseif looping_method == "imperfect_1"
        
        r1_to_load, r1_to_bess, r1_excess, r2_to_load, r2_to_bess, r2_excess, bess_to_load, bess_soc, util_to_load, util_to_bess = dispatch_method_1(result, realized_pv_prod_factor[idx]*pv_kw, 0.0, bess_kwh, soc_init_fraction, charge_efficiency, discharge_efficiency, hours_per_time_step)
        grid_power = max(util_to_load + util_to_bess, 0)
        
    elseif looping_method == "drts"
        
        # Only considers a grid-connected systems with PV, battery, and load for now
        pv_to_load = result["PV"]["electric_to_load_series_kw"][1]
        pv_to_battery = result["PV"]["electric_to_storage_series_kw"][1]
        pv_excess = result["PV"]["electric_to_grid_series_kw"][1] + result["PV"]["electric_curtailed_series_kw"][1]
        
        battery_to_load = result["ElectricStorage"]["storage_to_load_series_kw"][1]
        battery_soc = result["ElectricStorage"]["soc_series_fraction"][1]
        grid_to_load = result["ElectricUtility"]["electric_to_load_series_kw"][1]
        grid_to_battery = result["ElectricUtility"]["electric_to_storage_series_kw"][1]

        grid_consumption = max(grid_to_load + grid_to_battery, 0)  
        
        try
            # 1. Pset (PV Setpoint):
            if allow_export
                Pset = pv_to_load + pv_to_battery + pv_excess
            else
                Pset = pv_to_load + pv_to_battery
            end
            
            # 2. B_set (Battery Setpoint): -ve for charging
            B_set = battery_to_load - (pv_to_battery + grid_to_battery)

            # 3. Pack the two Float64 values as big-endian (>dd)
            io = IOBuffer()
            write_be_float32(io, Float32(Pset))
            write_be_float32(io, Float32(B_set))    
            message = take!(io)
            
            # Currently assuming every run will take less than 30 seconds
            now_ns = time_ns()
            if now_ns < next_send_time
                sleep((next_send_time - now_ns) / 1e9)
            end

            # 4. Send the message
            sock = UDPSocket()
            bind(sock, ip"10.81.15.151", LOCAL_PORT)
            
            send(sock, IPv4(REMOTE_IP), REMOTE_PORT, message)
            close(sock)
            sent_at = time_ns()
            println(Dates.format(now(), "yyyy-mm-dd HH:MM:SS.sss"))

            println("Sent $(length(message)) bytes from local port $LOCAL_PORT")
            
            # Schedule the next send 30 seconds later
            next_send_time += PERIOD_NS
    
            run_count += 1
            println("\nRun $run_count: Pset = $(round(Pset, digits=4)), B_set = $(round(B_set, digits=4))")
        catch e
            println("\nAn error occurred: $e")
        end

        # Add listening code
        sock = UDPSocket()
        bind(sock, ip"10.81.15.151", LOCAL_PORT)
        
        data = recv(sock)
        println("Received $(length(data)) bytes")
        values = unpack_float32_be(data)
        close(sock)
        
        if length(values) == 9
            Pset_pv = values[1]*1000
            Qset_pv = values[2]*1000

            Pset_bess = values[3]*1000
            Qset_bess = values[4]*1000

            Pset_load = values[5]*1000
            Qset_load = values[6]*1000

            P_grid = values[7]*1000
            Q_grid = values[8]*1000

            BESS_soc = values[9]/100

            println("\nPset_bess = ", Pset_bess)
            println("Pset_pv = ", Pset_pv)
            println("Pset_load = ", Pset_load)
            println("Qset_bess = ", Qset_bess)
            println("Qset_pv = ", Qset_pv)
            println("Qset_load = ", Qset_load)
            println("\nP_grid = ", P_grid)
            println("Q_grid = ", Q_grid)
            println("BESS_soc = ", BESS_soc)

            push!(drts_signals["PV"]["Pset_pv"], Pset_pv)
            push!(drts_signals["PV"]["Qset_pv"], Qset_pv)
            push!(drts_signals["Battery"]["Pset_bess"], Pset_bess)
            push!(drts_signals["Battery"]["Qset_bess"], Qset_bess)
            push!(drts_signals["Battery"]["BESS_soc"], BESS_soc)
            push!(drts_signals["Load"]["Pset_load"], Pset_load)
            push!(drts_signals["Load"]["Qset_load"], Qset_load)
            push!(drts_signals["Grid"]["P_grid"], P_grid)
            push!(drts_signals["Grid"]["Q_grid"], Q_grid)
            
            util_to_load, util_to_bess, r1_to_load, r1_to_bess, bess_to_load, r1_excess = 
                                                                    split_power_flows(Pset_pv, Pset_bess, Pset_load, P_grid)
            bess_soc = BESS_soc
            grid_power = P_grid
            
        else
            println("Expected 9 Float32 values, got $(length(values))")
            println("Decoded values = ", values)
        end
        
    else
        print("Undefined Looping Method")
        break
    end
   
    # Process time series results for the timestep executed 
    push!(dispatch_results["PV"]["electric_to_load_series_kw"], r1_to_load)
    push!(dispatch_results["PV"]["electric_to_storage_series_kw"], r1_to_bess)
    if allow_export
        push!(dispatch_results["PV"]["electric_to_grid_series_kw"], r1_excess)
        push!(dispatch_results["PV"]["electric_curtailed_series_kw"], 0)
    else
        push!(dispatch_results["PV"]["electric_to_grid_series_kw"], 0)
        push!(dispatch_results["PV"]["electric_curtailed_series_kw"], r1_excess)
    end
    
    push!(dispatch_results["ElectricStorage"]["storage_to_load_series_kw"], bess_to_load)
    push!(dispatch_results["ElectricStorage"]["soc_series_fraction"], round(bess_soc, digits=6))

    push!(dispatch_results["ElectricUtility"]["electric_to_load_series_kw"], util_to_load)
    push!(dispatch_results["ElectricUtility"]["electric_to_storage_series_kw"], util_to_bess) 

    push!(dispatch_results["ElectricLoad"]["load_series_kw"], result["ElectricLoad"]["load_series_kw"][1])
    
    # Calculate running sums of relevant values
    energy_cost_kwh += grid_power * hours_per_time_step * post["ElectricTariff"]["energy_rates"][1]
    push!(cost_series, grid_power * hours_per_time_step * post["ElectricTariff"]["energy_rates"][1])
    
    soc_init_fraction = bess_soc

    if rate_scenario == "tou"
        # Find which time of use period the current timestep is in
        i = findfirst(==(demand_cost[idx]), tou_demand_rates)
        tou_previous_peak_demands[i] = max(grid_power, tou_previous_peak_demands[i])
        
        # Calculate the total TOU demand charge at the end of each month
        # Currently, MPC returns demand charge = 0 if run for less than 1 month
        if idx in month_transition_timesteps
            append!(monthly_demand_cost, tou_previous_peak_demands .* tou_demand_rates)
            push!(tou_peaks_by_month, tou_previous_peak_demands)
            tou_previous_peak_demands = zeros(length(tou_demand_rates))
        end
    end
    
end

if save_results
    open(outputs_path * "/" * filename * "_reopt_dispatch.json","w") do f
      JSON.print(f, dispatch_results)
    end
end

if looping_method == "drts"
    close(sock)
    
    if save_results
        open(outputs_path * "/" * filename * "_drts_raw.json","w") do f
          JSON.print(f, dispatch_results)
        end
    end
end

print("\n\nFinished loop\n")

2026-03-20 15:42:58.983
Sent 8 bytes from local port 10001

Run 1: Pset = 192.014, B_set = -500.0
Received 36 bytes

Pset_bess = -0.001555009
Pset_pv = -0.001555009
Pset_load = -0.001555009
Qset_bess = 2.686551e-20
Qset_pv = 2.686551e-20
Qset_load = 2.686551e-20

P_grid = 0.004665026
Q_grid = -2.3152587e-10
BESS_soc = 0.20000002


[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.011 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.012 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.


2026-03-20 15:43:28.923
Sent 8 bytes from local port 10001

Run 2: Pset = 158.675, B_set = -500.0
Received 36 bytes

Pset_bess = -500.0
Pset_pv = 192.014
Pset_load = -500.0
Qset_bess = 1.0000025
Qset_pv = 0.9999991
Qset_load = 1.0000025

P_grid = 807.986
Q_grid = -3.000004
BESS_soc = 0.2396749


[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.012 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.


2026-03-20 15:43:58.925
Sent 8 bytes from local port 10001

Run 3: Pset = 209.349, B_set = -500.0
Received 36 bytes

Pset_bess = -500.0
Pset_pv = 192.014
Pset_load = -500.0
Qset_bess = 1.0000001
Qset_pv = 1.0
Qset_load = 1.0000001

P_grid = 807.986
Q_grid = -3.0000002
BESS_soc = 0.26731625


...

[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.011 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.


2026-03-20 16:15:28.898
Sent 8 bytes from local port 10001

Run 66: Pset = 203.34, B_set = -126.299
Received 36 bytes

Pset_bess = -131.561
Pset_pv = 195.261
Pset_load = -500.0
Qset_bess = 1.0000006
Qset_pv = 0.9999992
Qset_load = 1.0000023

P_grid = 436.30002
Q_grid = -3.0000021
BESS_soc = 0.8191387


[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.01 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.


2026-03-20 16:15:58.897
Sent 8 bytes from local port 10001

Run 67: Pset = 205.573, B_set = -119.248
Received 36 bytes

Pset_bess = -130.81137
Pset_pv = 195.88884
Pset_load = -500.00006
Qset_bess = 0.9999997
Qset_pv = 1.0000004
Qset_load = 0.9999992

P_grid = 434.92258
Q_grid = -2.9999993
BESS_soc = 0.82638097


[ Info: Model built. Optimizing...
┌ Info: MPC solved with 
└   termination_status(m) = OPTIMAL::TerminationStatusCode = 1
[ Info: Solving took 0.011 seconds.
[ Info: Results processing took 0.001 seconds.
[ Info: Results processing took 0.001 seconds.


In [ ]:
if !isempty(monthly_demand_cost)
    total_demand = sum(monthly_demand_cost)
else
    total_demand = 0
end

print("\nEnergy cost (\$): ", format(round(energy_cost_kwh, digits=3), commas=true))
print("\nDemand cost (\$): ", format(total_demand, commas=true))
print("\nTotal bill (\$): ", format(energy_cost_kwh + total_demand, commas=true))

grid_purchase = sum(dispatch_results["ElectricUtility"]["electric_to_load_series_kw"]) +
                sum(dispatch_results["ElectricUtility"]["electric_to_storage_series_kw"])
print("\n\nGrid purchases (kWh): ", format(round(grid_purchase, digits=3), commas=true))
# print("\n\nPercent RE: ", (sum(load) - grid_purchase)/sum(load)*100)

if stopping_ts == 8760
    pv_re = sum((pv_prod_factor * pv_kw) - 
            (dispatch_results["PV"]["electric_to_storage_series_kw"] * (1 - charge_efficiency*discharge_efficiency)) -
            (dispatch_results["PV"]["electric_curtailed_series_kw"]))

    print("\nCalculated percent RE: ", ((pv_re)/sum(load)*100))
end

battery_throughput = (sum(dispatch_results["PV"]["electric_to_storage_series_kw"]) + 
                  sum(dispatch_results["ElectricUtility"]["electric_to_storage_series_kw"])) * charge_efficiency

BESS_out = (sum(dispatch_results["ElectricStorage"]["storage_to_load_series_kw"])) / discharge_efficiency

print("\n\nBattery throughput (kWh in): ", format(round(battery_throughput, digits=3), commas=true))
print("\nBattery throughput (kWh out): ", format(round(BESS_out, digits=3), commas=true))

In [ ]:
df = DataFrame()

# df[!, "pv_prod_factor"] = pv_prod_factor
df[!, "pv_to_load_series_kw"] = dispatch_results["PV"]["electric_to_load_series_kw"]
df[!, "pv_to_storage_series_kw"] = dispatch_results["PV"]["electric_to_storage_series_kw"]
df[!, "pv_to_grid_series_kw"] = dispatch_results["PV"]["electric_to_grid_series_kw"]
df[!, "pv_curtailed_production_series_kw"] = dispatch_results["PV"]["electric_curtailed_series_kw"]

df[!, "storage_to_load_series_kw"] = dispatch_results["ElectricStorage"]["storage_to_load_series_kw"]
df[!, "soc_series_fraction"] = dispatch_results["ElectricStorage"]["soc_series_fraction"]

df[!, "grid_to_load_series_kw"] = dispatch_results["ElectricUtility"]["electric_to_load_series_kw"]
df[!, "grid_to_battery_series_kw"] = dispatch_results["ElectricUtility"]["electric_to_storage_series_kw"]

df[!, "cost_per_timestep"] = cost_series

if save_results
    CSV.write(outputs_path * "/" * filename * "_reopt_dispatch.csv", df);
end

In [ ]:
df = DataFrame()

df[!, "Pset_pv"] = drts_signals["PV"]["Pset_pv"]
df[!, "Pset_bess"] = drts_signals["Battery"]["Pset_bess"]

df[!, "P_grid"] = drts_signals["Grid"]["P_grid"]
df[!, "Pset_load"] = drts_signals["Load"]["Pset_load"]
df[!, "BESS_soc"] = drts_signals["Battery"]["BESS_soc"]

df[!, "Qset_pv"] = drts_signals["PV"]["Qset_pv"]
df[!, "Qset_bess"] = drts_signals["Battery"]["Qset_bess"]
df[!, "Qset_load"] = drts_signals["Load"]["Qset_load"]
df[!, "Q_grid"] = drts_signals["Grid"]["Q_grid"]

if save_results
    CSV.write(outputs_path * "/" * filename * "_drts_raw.csv", df);
end

In [8]:
using Dates

# First send time = now
const PERIOD_NS   = 30_000_000_000 # 30 sec
const TWO_MIN_NS   = 120_000_000_000 # 2 min

start_time = time_ns()
stop_time  = start_time + TWO_MIN_NS

next_send_time = start_time #time_ns()

while time_ns() < stop_time
    # ---------------------------------
    # 1. Wait until scheduled send time
    # ---------------------------------
    now_ns = time_ns()
    if now_ns < next_send_time
        sleep((next_send_time - now_ns) / 1e9)
    end

    # ---------------------------------
    # 2. Send exactly on schedule
    # ---------------------------------
#     send(sock, REMOTE_IP, REMOTE_PORT, next_message)
    sent_at = time_ns()
#     println(Dates.format(unix2datetime(Base.time_ns() / 1e9), "yyyy-mm-dd HH:MM:SS.sss"))
    println(Dates.format(now(), "yyyy-mm-dd HH:MM:SS.sss"))
#     println("Sent Pset=$(next_Pset), B_set=$(next_B_set) at $(sent_at / 1e9) s")

    # Schedule the next send 30 seconds later
    next_send_time += PERIOD_NS
    
#     # ---------------------------------
#     # 3. Receive response
#     #    This blocks until a packet arrives
#     # ---------------------------------
#     data = recv(sock)
#     latest_received = unpack_float32_be(data)

#     println("Received values: ", latest_received)

#     # ---------------------------------
#     # 4. Run optimization using new signal
#     # ---------------------------------
#     next_Pset, next_B_set = run_optimization(latest_received)
#     next_message = pack_two_float32_be(next_Pset, next_B_set)

#     # ---------------------------------
#     # 5. If optimization/receive took longer than 30 s,
#     #    don't go backwards in time
#     # ---------------------------------
#     if time_ns() > next_send_time
#         println("Warning: cycle overran 30 seconds; next send will be late.")
#         next_send_time = time_ns()
#     end
end

2026-03-18 18:08:30.856
2026-03-18 18:09:00.823
2026-03-18 18:09:30.812
2026-03-18 18:10:00.811
2026-03-18 18:10:30.822


In [17]:
            sock = UDPSocket()
            bind(sock, ip"10.81.15.151", LOCAL_PORT) 

data = recv(sock)
        println("Received $(length(data)) bytes")
        values = unpack_float32_be(data)
        close(sock)
        
            Pset_pv = values[1]*1000
            Qset_pv = values[2]*1000

            Pset_bess = values[3]*1000
            Qset_bess = values[4]*1000

            Pset_load = values[5]*1000
            Qset_load = values[6]*1000

            P_grid = values[7]*1000
            Q_grid = values[8]*1000

            BESS_soc = values[9]/100

            println("\nPset_bess = ", Pset_bess)
            println("Pset_pv = ", Pset_pv)
            println("Pset_load = ", Pset_load)
            println("Qset_bess = ", Qset_bess)
            println("Qset_pv = ", Qset_pv)
            println("Qset_load = ", Qset_load)
            println("\nP_grid = ", P_grid)
            println("Q_grid = ", Q_grid)
            println("BESS_soc = ", BESS_soc)

Received 36 bytes

Pset_bess = -255.382
Pset_pv = 205.401
Pset_load = -500.0
Qset_bess = 1.0000012
Qset_pv = 0.9999991
Qset_load = 1.0000025

P_grid = 549.981
Q_grid = -3.0000029
BESS_soc = 0.74304307
